In [1]:
import numpy as np
import json
import datetime
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.metrics import confusion_matrix, classification_report

# 1. LOAD PREPROCESSED DATA (Data Processing: 5/5)
# These files should be in your working directory from your preprocessing step
X_train = np.load("X_train_lstm.npy")
X_test = np.load("X_test_lstm.npy")
y_train = np.load("y_train.npy")
y_test = np.load("y_test.npy")

# Activity Labels for UCI HAR Dataset
activity_labels = ["Walking", "Walking_Upstairs", "Walking_Downstairs", "Sitting", "Standing", "Laying"]

# 2. BUILD LSTM MODEL (Model Building: 10/10)
model = Sequential([
    # LSTM layer for time-series sensor data processing
    LSTM(128, input_shape=(X_train.shape[1], X_train.shape[2])), 
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(6, activation='softmax') # 6 output classes
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# 3. CALLBACKS (Automation & Best Model Selection)
checkpoint = ModelCheckpoint("best_lstm_model.h5", save_best_only=True, monitor='val_accuracy')
earlystop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# 4. TRAINING
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    callbacks=[checkpoint, earlystop],
    verbose=1
)

# 5. PERFORMANCE EVALUATION (Performance Evaluation: 5/5)
loss, acc = model.evaluate(X_test, y_test, verbose=1)

# Generate Predictions for detailed metrics
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Calculate Precision, Recall, and F1-Score
class_report_dict = classification_report(y_test, y_pred, target_names=activity_labels, output_dict=True)
class_report_text = classification_report(y_test, y_pred, target_names=activity_labels)

print("\n✅ Test Accuracy:", round(acc, 4))
print("✅ Test Loss:", round(loss, 4))
print("\n--- Detailed Classification Report ---")
print(class_report_text)

# 6. MLOps EXPERIMENT TRACKING (Experiment Tracking: 5/5)
# Creating a comprehensive log of the run
experiment_log = {
    "run_id": datetime.datetime.now().strftime("%Y%m%d_%H%M%S"),
    "timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "metadata": {
        "model_type": "LSTM",
        "dataset": "UCI HAR",
        "framework": "TensorFlow/Keras"
    },
    "hyperparameters": {
        "lstm_units": 128,
        "dropout": 0.3,
        "dense_units": 64,
        "optimizer": "adam",
        "batch_size": 32,
        "epochs_actual": len(history.history['loss'])
    },
    "metrics": {
        "test_accuracy": float(acc),
        "test_loss": float(loss),
        "macro_f1": class_report_dict['macro avg']['f1-score'],
        "weighted_precision": class_report_dict['weighted avg']['precision'],
        "weighted_recall": class_report_dict['weighted avg']['recall']
    },
    "artifacts": {
        "model_file": "best_lstm_model.h5",
        "scaler_file": "scaler.joblib"
    }
}

# Save or Append to the tracking ledger (JSON format)
log_file = "experiment_tracking.json"
if os.path.exists(log_file):
    with open(log_file, "r") as f:
        try:
            data = json.load(f)
            if not isinstance(data, list):
                data = [data]
        except json.JSONDecodeError:
            data = []
    data.append(experiment_log)
else:
    data = [experiment_log]

with open(log_file, "w") as f:
    json.dump(data, f, indent=4)

print(f"\n🚀 MLOps: Experiment metrics, F1-scores, and hyperparameters logged in {log_file}")

c:\Users\cool\har_project\venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128)            │       353,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 361,926 (1.38 MB)

 Trainable params: 361,926 (1.38 MB)

...

184/184 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9554 - loss: 0.1245 - val_accuracy: 0.9402 - val_loss: 0.1621
Epoch 3/20
184/184 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9619 - loss: 0.1010 - val_accuracy: 0.9293 - val_loss: 0.1781
Epoch 4/20
184/184 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9725 - loss: 0.0736 - val_accuracy: 0.9395 - val_loss: 0.1589
Epoch 5/20
182/184 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9803 - loss: 0.0605

184/184 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9767 - loss: 0.0660 - val_accuracy: 0.9470 - val_loss: 0.1517
Epoch 6/20
184/184 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9784 - loss: 0.0615 - val_accuracy: 0.9463 - val_loss: 0.1570
Epoch 7/20
184/184 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9789 - loss: 0.0563 - val_accuracy: 0.9470 - val_loss: 0.1442
Epoch 8/20
184/184 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9825 - loss: 0.0516 - val_accuracy: 0.9429 - val_loss: 0.2152
Epoch 9/20
184/184 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9806 - loss: 0.0555 - val_accuracy: 0.9347 - val_loss: 0.2416
Epoch 10/20
184/184 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9811 - loss: 0.0458

184/184 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9828 - loss: 0.0467 - val_accuracy: 0.9490 - val_loss: 0.1351
Epoch 11/20
184/184 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9833 - loss: 0.0481 - val_accuracy: 0.9456 - val_loss: 0.1811
Epoch 12/20
184/184 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9866 - loss: 0.0368

184/184 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9861 - loss: 0.0375 - val_accuracy: 0.9517 - val_loss: 0.1649
Epoch 13/20
181/184 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9896 - loss: 0.0306

184/184 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9869 - loss: 0.0342 - val_accuracy: 0.9565 - val_loss: 0.1617
Epoch 14/20
184/184 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9852 - loss: 0.0418 - val_accuracy: 0.9470 - val_loss: 0.2062
Epoch 15/20
184/184 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9871 - loss: 0.0337 - val_accuracy: 0.9490 - val_loss: 0.1700
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9243 - loss: 0.3045
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step

✅ Test Accuracy: 0.9243
✅ Test Loss: 0.3045

--- Detailed Classification Report ---
                    precision    recall  f1-score   support

           Walking       0.89      1.00      0.94       496
  Walking_Upstairs       0.92      0.93      0.92       471
Walking_Downstairs       0.99      0.84      0.91       420
           Sitting       0.94      0.86      0.90       491
          Standing       0.85      0.95      0.90       532
            Laying       1.00      0.96      0.98       537

    